# Indic Document Intelligence with Kimi K3 on Amazon Bedrock

This notebook shows how to use **Kimi K3** on **Amazon Bedrock** to read scanned Hindi documents and turn them into structured data, using the **India cross-Region inference profile** (`in.moonshotai.kimi-k3`) so that inference stays within India (Mumbai `ap-south-1` and Hyderabad `ap-south-2`).

You will:

1. Send a scanned Hindi electricity bill to Kimi K3 in a single [Chat Completions API](https://docs.aws.amazon.com/bedrock/latest/userguide/inference-chat-completions.html) call on the `bedrock-runtime` endpoint, as the [Kimi K3 model card](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-moonshot-ai-kimi-k3.html) recommends, and get back a full transcription, structured fields as JSON, and an English summary.
2. Check the extracted fields against hand-labelled ground truth.
3. Ask follow-up questions about the same document in **Hinglish** (code-mixed Hindi-English).
4. Repeat the flow on a hand-filled Hindi KYC form, and on any document of your own using the schema-free `generic` mode.

> **Note:** All documents in `sample_data/` are **synthetic**. Every organisation, person, address and number is fictional, and identity numbers are masked. Do not use real personal documents with this sample unless you have the right to process them.

## Prerequisites

- An AWS account with access to Kimi K3 in Amazon Bedrock in `ap-south-1`. See [Access Amazon Bedrock foundation models](https://docs.aws.amazon.com/bedrock/latest/userguide/model-access.html).
- IAM permission for `bedrock:InvokeModel` on the `in.moonshotai.kimi-k3` inference profile and the underlying foundation model in both India Regions, plus `bedrock:CallWithBearerToken`. The sample turns your AWS credentials into a short-term Amazon Bedrock API key. See the README for an example policy.
- Python 3.10 or later, and AWS credentials configured for this environment.

## Setup

Installs the Python packages listed in `requirements.txt`.

In [1]:
# Install the sample's Python dependencies (OpenAI SDK, Bedrock token generator, Pillow).
%pip install -q --disable-pip-version-check -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


Imports the sample code, loads the settings (model ID, Region, reasoning effort) and creates the `pipeline` object that calls Kimi K3 on Amazon Bedrock.

In [2]:
import json
import sys
from pathlib import Path

# Make the sample's `indic_doc_intel` package (one folder up) importable from this notebook.
sys.path.insert(0, str(Path.cwd().parent))

from indic_doc_intel import IndicDocumentIntelligence, Settings
from indic_doc_intel.evaluation import compare_fields, load_expected

# Folder with the synthetic sample documents and their ground-truth labels.
SAMPLE_DATA = Path.cwd().parent / "sample_data"

# Defaults: model in.moonshotai.kimi-k3, Region ap-south-1, model-default reasoning effort (max).
# Override with KIMI_MODEL_ID, BEDROCK_REGION and KIMI_REASONING_EFFORT, or pass
# Settings(reasoning_effort="low") to trade a little quality for fewer output tokens.
settings = Settings.from_env()

# Creates the Chat Completions client for the bedrock-runtime endpoint. It turns your AWS
# credentials into a short-term Amazon Bedrock API key; no long-term key is needed.
pipeline = IndicDocumentIntelligence(settings)
print(f"Model: {settings.model_id}\nRegion: {settings.region}")

Model: in.moonshotai.kimi-k3
Region: ap-south-1


Defines two small display helpers used in later cells: `show_json` prints results, and `show_comparison` prints a field-by-field ground-truth check.

In [3]:
def show_json(data):
    """Pretty-print a dict, keeping Hindi text readable (ensure_ascii=False)."""
    print(json.dumps(data, ensure_ascii=False, indent=2))


def show_comparison(rows):
    """Print one line per field: OK if the extracted value matches ground truth, DIFF otherwise."""
    width = max(len(row["field"]) for row in rows)
    for row in rows:
        mark = "OK  " if row["match"] else "DIFF"
        print(
            f"{mark}  {row['field']:<{width}}  expected={row['expected']!r:<30} extracted={row['extracted']!r}"
        )
    # Summary line, e.g. "8/8 fields match ground truth"
    matched = sum(row["match"] for row in rows)
    print(f"\n{matched}/{len(rows)} fields match ground truth")

## Step 1: Look at the sample document

A synthetic electricity bill (बिजली बिल) from a fictional distribution company in Jaipur. Note that the dates and the units consumed use **Devanagari digits** (e.g. `०५/१०/२०२६`, `२८८`), while other numbers use Western digits, as is common on real Indian documents.

<img src="../sample_data/electricity_bill_hi.png" width="520" alt="Synthetic Hindi electricity bill">

## Step 2: Extract structured fields with Kimi K3

`pipeline.extract()` sends the image (placed before the text, as the model card recommends) together with an extraction prompt built from the `electricity_bill` schema in [`indic_doc_intel/schemas.py`](../indic_doc_intel/schemas.py). Kimi K3 reads the document and returns one JSON object containing the transcription, the requested fields (dates normalised to ISO format, Devanagari digits converted, names transliterated) and an English summary.

Sends the scanned Hindi bill to Kimi K3 and prints the structured fields it extracted.

In [4]:
bill_image = SAMPLE_DATA / "electricity_bill_hi.png"

# One call to Kimi K3: sends the image plus an extraction prompt built from the
# "electricity_bill" schema, and parses the JSON reply into a DocumentResult.
bill = pipeline.extract(bill_image, document_type="electricity_bill")

# Structured fields: ISO dates, Western digits, names transliterated into English.
show_json(bill.fields)

{
  "distribution_company": "Kalpanik Vidyut Vitran Nigam Ltd.",
  "consumer_name": "श्रीमती सुनीता देवी शर्मा",
  "consumer_name_english": "Shrimati Sunita Devi Sharma",
  "consumer_number": "410277893561",
  "address": "House No. 27, Gali No. 4, Shastri Nagar, Jaipur, Rajasthan - 302016",
  "billing_period": "2026-09-01 to 2026-09-30",
  "bill_date": "2026-10-05",
  "due_date": "2026-10-20",
  "units_consumed_kwh": 288,
  "amount_due_inr": 2217.2,
  "amount_after_due_date_inr": 2261.54
}


Shows the rest of the extraction result: the detected languages, the English summary and the start of the Hindi transcription.

In [5]:
# The same reply also contains the detected languages, an English summary
# and a full transcription of the document in its original script.
print("Detected languages:", bill.data["detected_languages"])
print("\nEnglish summary:\n", bill.english_summary)
print("\nTranscription (first 600 characters):\n", bill.data["transcription"][:600])

Detected languages: ['hi', 'en']

English summary:
 This is an electricity bill issued by Kalpanik Vidyut Vitran Nigam Ltd. to consumer Shrimati Sunita Devi Sharma for the billing period 01 September 2026 to 30 September 2026, with a consumption of 288 units. The total amount payable is INR 2,217.20, due by 20 October 2026; payment after the due date incurs a 2% late fee, raising the amount to INR 2,261.54. The bill warns that the electricity connection may be disconnected if two consecutive bills remain unpaid.

Transcription (first 600 characters):
 काल्पनिक विद्युत वितरण निगम लिमिटेड
Kalpanik Vidyut Vitran Nigam Ltd. (काल्पनिक संस्था — केवल प्रदर्शन हेतु)
ग्राहक सेवा: 1800-000-0000 | www.example.com

बिजली बिल / विद्युत उपभोग देयक

उपभोक्ता का नाम	श्रीमती सुनीता देवी शर्मा
उपभोक्ता संख्या	4102 7789 3561
पता	मकान नं. 27, गली नं. 4, शास्त्री नगर, जयपुर, राजस्थान – 302016
कनेक्शन श्रेणी	घरेलू (एकल फेज़), स्वीकृत भार 3 किलोवाट
बिल दिनांक	०५/१०/२०२६
बिल अवधि	०१/०९/२०२६ से ३०/०९/२०२६

मीट

## Step 3: Check against ground truth

`sample_data/expected/` holds hand-labelled values for each sample. Comparing against them is a simple way to catch regressions when you change prompts or models.

Scores the extraction against the hand-labelled ground truth for the bill.

In [6]:
# Compare the extracted fields with the hand-labelled values in
# sample_data/expected/electricity_bill_hi.json (numbers and spacing are normalised).
show_comparison(compare_fields(bill.fields, load_expected(bill_image)))

OK    consumer_name              expected='श्रीमती सुनीता देवी शर्मा'    extracted='श्रीमती सुनीता देवी शर्मा'
OK    consumer_number            expected='410277893561'                 extracted='410277893561'
OK    billing_period             expected='2026-09-01 to 2026-09-30'     extracted='2026-09-01 to 2026-09-30'
OK    bill_date                  expected='2026-10-05'                   extracted='2026-10-05'
OK    due_date                   expected='2026-10-20'                   extracted='2026-10-20'
OK    units_consumed_kwh         expected=288                            extracted=288
OK    amount_due_inr             expected=2217.2                         extracted=2217.2
OK    amount_after_due_date_inr  expected=2261.54                        extracted=2261.54

8/8 fields match ground truth


## Step 4: Ask follow-up questions in Hinglish

Many users in India type in Hinglish: Hindi words written in Latin script, mixed with English. `pipeline.ask()` continues the same conversation, so Kimi K3 still has the original image in context, and is asked to reply in the user's own language style.

Asks three Hinglish questions about the bill, with one Kimi K3 call per question, and prints each answer.

In [7]:
# Follow-up questions in Hinglish (Hindi written in Latin script, mixed with English).
questions = [
    "Bill kitne ka hai aur last date kab hai? Late hua toh kitna dena padega?",
    "Is mahine kitne units use hue aur meter number kya hai?",
    "Agar main bill pay nahi karun toh kya hoga?",
]

# Each ask() continues the extraction conversation, so the model still sees the bill image,
# and replies in the same style as the question (Hinglish in, Hinglish out).
for question in questions:
    print(f"Q: {question}\nA: {pipeline.ask(bill, question)}\n")

Q: Bill kitne ka hai aur last date kab hai? Late hua toh kitna dena padega?
A: Bill ₹2,217.20 ka hai, jisme 288 units ka energy charge, fixed charge aur electricity duty shaamil hain. Last date 20/10/2026 hai. Agar due date ke baad payment karte hain, toh 2% late surcharge ke saath ₹2,261.54 dena padega.



Q: Is mahine kitne units use hue aur meter number kya hai?
A: Is mahine 288 units use hue hain. Meter number JPR-55120934 hai.



Q: Agar main bill pay nahi karun toh kya hoga?
A: Document ke hisaab se, agar aap lagatar do bills ka payment nahi karte, toh aapka bijli connection kaata ja sakta hai.



## Step 5: A hand-filled KYC form

The second sample is a synthetic address-update / KYC form for a fictional cooperative bank, filled in by hand in blue ink, with tick boxes, a masked Aadhaar number and a signature. The same pipeline is used with the `kyc_form` schema.

<img src="../sample_data/kyc_form_hi.png" width="520" alt="Synthetic Hindi KYC form">

Extracts the hand-filled KYC form with the `kyc_form` schema, prints the fields, and checks them against ground truth.

In [8]:
kyc_image = SAMPLE_DATA / "kyc_form_hi.png"

# Same pipeline, different schema: "kyc_form" defines the fields to extract from this form.
kyc = pipeline.extract(kyc_image, document_type="kyc_form")

show_json(kyc.fields)
print()
# Score against sample_data/expected/kyc_form_hi.json
show_comparison(compare_fields(kyc.fields, load_expected(kyc_image)))

{
  "applicant_name": "राजेश कुमार यादव",
  "applicant_name_english": "Rajesh Kumar Yadav",
  "father_or_spouse_name_english": "Ramprasad Yadav",
  "date_of_birth": "1988-03-14",
  "gender": "male",
  "mobile_number": "9876543210",
  "address": "Flat No. 302, Sai Kripa Apartment, Vibhuti Khand, Gomti Nagar, Lucknow, Uttar Pradesh - 226010",
  "pin_code": "226010",
  "id_document_type": "Aadhaar (masked)",
  "id_document_number": "XXXX XXXX 4821",
  "declaration_signed": true
}

OK    applicant_name      expected='राजेश कुमार यादव'             extracted='राजेश कुमार यादव'
OK    date_of_birth       expected='1988-03-14'                   extracted='1988-03-14'
OK    gender              expected='male'                         extracted='male'
OK    mobile_number       expected='9876543210'                   extracted='9876543210'
OK    pin_code            expected='226010'                       extracted='226010'
OK    id_document_number  expected='XXXX XXXX 4821'               extracted=

Asks one Hinglish question about the KYC form and prints the answer.

In [9]:
# A Hinglish follow-up question about the KYC form (continues the KYC conversation).
print(
    pipeline.ask(
        kyc, "Is form mein naya address kya likha hai aur kaunsa ID proof diya hai?"
    )
)

Form mein naya address yeh likha hai: Flat No. 302, Sai Kripa Apartment, Vibhuti Khand, Gomti Nagar, Lucknow, Uttar Pradesh – 226010.

ID proof ke roop mein masked Aadhaar diya gaya hai, jiska document number XXXX XXXX 4821 hai.


## Step 6: Bring your own document (schema-free mode)

With `document_type="generic"`, no schema is supplied and Kimi K3 chooses the most important key-value pairs itself. This is useful for exploring a new document type before you write a schema for it. Point `my_image` at your own (non-sensitive) scanned document to try it.

Runs schema-free extraction. Point `my_image` at your own non-sensitive scan to try a new document type.

In [10]:
my_image = SAMPLE_DATA / "electricity_bill_hi.png"  # replace with your own image path

# "generic" sends no schema: Kimi K3 picks the document type and the key fields itself.
generic = pipeline.extract(my_image, document_type="generic")
print("Document type:", generic.data["document_type"])
show_json(generic.fields)

Document type: Electricity bill
{
  "consumer_name": "श्रीमती सुनीता देवी शर्मा",
  "consumer_number": "410277893561",
  "address": "मकान नं. 27, गली नं. 4, शास्त्री नगर, जयपुर, राजस्थान – 302016",
  "connection_category": "घरेलू (एकल फेज़), स्वीकृत भार 3 किलोवाट",
  "bill_date": "2026-10-05",
  "billing_period_start": "2026-09-01",
  "billing_period_end": "2026-09-30",
  "meter_number": "JPR-55120934",
  "total_units_consumed": 288,
  "total_amount_due": 2217.2,
  "due_date": "2026-10-20",
  "amount_after_due_date": 2261.54
}


## Token usage

Each `DocumentResult` adds up the token usage reported by the Chat Completions API across extraction and follow-up questions. `completion_tokens` includes `reasoning_tokens`, and `prompt_tokens` includes `cached_prompt_tokens`, which are billed at the lower cache-read rate. Multiply by the current Kimi K3 rates on the [Amazon Bedrock pricing page](https://aws.amazon.com/bedrock/pricing/) to estimate cost.

Prints the token usage of each document so you can estimate cost.

In [11]:
# Each result adds up the tokens used by its extraction call and any follow-up questions.
for name, result in [
    ("electricity_bill", bill),
    ("kyc_form", kyc),
    ("generic", generic),
]:
    print(f"{name:<17} {result.usage}")

electricity_bill  {'prompt_tokens': 17946, 'completion_tokens': 2304, 'total_tokens': 20250, 'cached_prompt_tokens': 16213, 'reasoning_tokens': 1086}
kyc_form          {'prompt_tokens': 8004, 'completion_tokens': 3471, 'total_tokens': 11475, 'cached_prompt_tokens': 3357, 'reasoning_tokens': 2286}
generic           {'prompt_tokens': 3214, 'completion_tokens': 3157, 'total_tokens': 6371, 'cached_prompt_tokens': 0, 'reasoning_tokens': 2087}


## Clean up

This notebook does not create any AWS resources. You are charged only for the Amazon Bedrock tokens consumed by the calls above.

## Next steps

- Add a schema for your own document type in `indic_doc_intel/schemas.py`, plus a ground-truth file in `sample_data/expected/`.
- Try documents in other Indic languages (Marathi, Tamil, Bengali and others) with the `generic` schema.
- For production workloads, add human review for low-confidence fields (see `uncertain_fields` in each result) and store results in a system such as Amazon DynamoDB or Amazon S3.